# SPOD to Sharepoint integration

- Prerequisites: 
  - [Office365-REST-Python-Client](https://pypi.org/project/Office365-REST-Python-Client/) installed. `pip install Office365-REST-Python-Client`
  - Anaconda packages: `pandas, openpyxl`
- Access to a [list(s)](https://support.microsoft.com/en-us/office/introduction-to-lists-0a1c3ace-def0-44af-b225-cfa8d92c52d7) in Sharepoint / Office365.

## Structure

1. Define mapping between SPOD (json) and columns in the list
1. Ensure list columns conform mapping. Automatically update if need.
1. Scan current content and preserve it in a table (Pandas)
1. Define and show changeset that will be applied
1. Upload changes
1. Apply changes via execute_query()


## Configuration

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

CONFIGURATION = 'sharepoint-bossard-sandbox.yaml'
CONFIGURATION = 'sharepoint-cop-mig.yaml'

apply_changes = True

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
configfile = Path(CONFIGURATION)
assert configfile.is_file(), f"Cannot find configuration file '{configfile.resolve()}'"

with open(configfile, 'r') as src:
    configuration = yaml.safe_load(src)
assert configuration['sharepoint'] is not None
spconf = configuration['sharepoint']
print(f"Loaded configuration for sharepoint acces with user '{spconf['credentials']['username']}' from {configfile}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.WARNING)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.sharepoint.list_publisher import update_structure, update_content, load_content

### Check SPOD

In [ ]:
spod_file = Path(configuration['spod'])
assert spod_file.is_file(), f"SPOD source missing: {spod_file.resolve()}"

In [ ]:
with open(spod_file, 'r') as src:
    spod = json.load(src)
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
mapdict = {}
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    print(f"- {key}: {len(spod[key])}")
    mapdict[key] = entry[key]['title']

## Access to Sharepoint site using Office365-REST-Python-Client library

In [ ]:
try:
    from office365.sharepoint.lists.list import List
    from office365.runtime.auth.user_credential import UserCredential
    from office365.sharepoint.client_context import ClientContext
except:
    print("Office356 API library is missing. Install it with 'pip install Office365-REST-Python-Client'")
    raise

In [ ]:
credentials = UserCredential(spconf['credentials']['username'], spconf['credentials']['password'])
ctx = ClientContext(spconf['site']).with_credentials(credentials)

In [ ]:
lists_available = ctx.lists.get().execute_query()
assert len(lists_available) > 0, f"Expecting more than 0 lists"
print(f"Found {len(lists_available)} lists in site {spconf['site']}")
print(f"Mapping exists for {len(mapdict)} tables: {list(mapdict.values())}")
for spl in lists_available:
    mapped = '✅' if spl.title in mapdict.values() else ''
    print(f"- {spl.title} {mapped}")

## Translation shortcut tr

In [ ]:
def tr(item, lang: str = 'en'):
    res = inner_tr(item, lang)
    if isinstance(res, dict):
        res = tr(res, lang)
    assert res is None or isinstance(res, str), f"Result is of type {type(res)}"
    return res

def inner_tr(item, lang) -> str:
    if isinstance(item, dict) and len(item) > 0:
        translation = item.get(lang)
        if translation is not None:
            return translation
        else:
            return next(iter(item.values()))
    if isinstance(item, str):
        return item
    
    if isinstance(item, list) and len(item) > 0:
        return tr(item[0])
    return ''

## List structure definition

In [ ]:
mappings = {}

### Defintion of the 'Entity' list

In [ ]:
import html
"""https://fyayc.sharepoint.com/sites/CoPInformation-Data-Governance/SiteAssets/Forms/AllItems.aspx?
    id=%2Fsites%2FCoPInformation%2DData%2DGovernance%2FSiteAssets%2Fsandbox%2Fdiagrams%2FDIAG698%2Esvg&
    parent=%2Fsites%2FCoPInformation%2DData%2DGovernance%2FSiteAssets%2Fsandbox%2Fdiagrams"""
def to_url_list(diagrams: list) -> str:
    result = [ "<div>"]
    for diagkey in diagrams:
        base = spconf['site'] + '/' 'SiteAssets/Forms/AllItems.aspx?id='
        target = '/sites/CoPInformation-Data-Governance/' + spconf['assets']['svg-diagrams'] + '/' + diagkey + '.svg'
        sid = html.escape(target)
        parent = "&parent=."
        title = spod['diagrams'][diagkey]['name']
        result.append(f"""<a href="{base}{sid}{parent}">{title}</a>""")
        result.append(", ")
    result = result[:-1]
    result.append("<div>")
    return ''.join(result)

In [ ]:
entity_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {}}),
    ('Synonyms', {'value': lambda e: tr(e['synonyms']), 'properties': {}}),
    ('Diagrams', {'value': lambda e: to_url_list(e['diagrams+']), 'properties': { 
        'FieldTypeKind': 3, 
        'TypeAsString': 'Note',
    }
    }),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'EnforceUniqueValues': True,
        'Filterable': True,
        'Sortable': True,
        'Indexed': True,
        'FieldTypeKind': 2,
        'Required': True,
    }}),
    #   ('Documentation Link', {'value': lambda e: './bla.html', 'properties': {'FieldType': 11}}),
]
mappings['entities'] = entity_mapping

### Definition of the 'Attribute' list
This list contains **all** attributes of the IM

In [ ]:
attribute_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {
        'Description': 'Business description of the attribute',
        'FieldTypeKind': 3,
        'FieldType': 'SP.FieldMultilineText'
    }}),
    ('Type', {'value': lambda a: a['type+'], 'properties': {
        'Description': 'Datatype of the attribute',
        'Required': False,
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'Required': True,
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
]
mappings['attributes'] = attribute_mapping

### Definition of the 'System' list

In [ ]:
systems_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {
        'Description': 'Description of the system',
        'FieldTypeKind': 3,
        'FieldType': 'SP.FieldMultilineText'
    }}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'Required': True,
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
]
mappings['systems'] = systems_mapping

# Preparation steps

## Preparing list structure 

In [ ]:
ctx.clear()

In [ ]:
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        sp_list = ctx.lists.get_by_title(entry[key]['title'])
        result = update_structure(sp_list, mapping, direct_write=apply_changes)
        print(f"Result:\n{os.linesep.join(result)}")
        
        fields = sp_list.fields.get().execute_query()
        has_key = False
        for field in fields:
            name = field.properties['EntityPropertyName']
            if 'Key' == name:
                has_key = True
        assert has_key
    else:
        logging.warning(f"No mapping for {key}")

In [ ]:
ctx.execute_query()

# Synchronize content
1. Read content, store it to local backup
2. Apply changes if any

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
spconf['lists']

In [ ]:
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        print(f"Processing list '{entry[key]['title']}'")
        sp_list = ctx.lists.get_by_title(entry[key]['title']).get().execute_query()
        
        content = load_content(sp_list.items)
        print(f"List '{entry[key]['title']}' currently contains {len(content)} entries. SPOD contains {len(spod[key])}")
        new, updated, deleted = update_content(sp_list, mapping, spod[key], content, direct_write=True)
        print(f" Creating: {len(new)}, updating: {len(updated)}, deleting: {len(deleted)} - executing {len(list(iter(ctx.pending_request())))} requests")
        sp_list.execute_query()
    else:
        logging.warning(f"No mapping for {key}")

In [ ]:
ctx.execute_query()

## Verify values written

In [ ]:
progressbar = None

def progress(items_read):
    progressbar.update(items_read)


for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        sp_list = ctx.lists.get_by_title(entry[key]['title']).get().execute_query()
        items = sp_list.items
        count = sp_list.properties.get('ItemCount')
        with tqdm(total=count) as progressbar:
            items._page_size = 50
            items.page_loaded += progress

            print(f"Updating mapping of {entry[key]['title']} (SPOD <-> Sharepoint List) {count}")
            content = load_content(items)
            for spod_key, item in content.items():
                sp_id = item.properties['ID']
                spod[key][spod_key]['sharepoint'] = sp_id
            progressbar.update(count)
            progressbar.close()

# Publish diagrams

In [ ]:
from office365.sharepoint.folders.folder import Folder

In [ ]:
destination = spconf['assets']['svg-diagrams'] 
print(f"Publishing {len(spod['diagrams'])} diagrams to {spconf['site']}/{destination}")

In [ ]:
folder = ctx.web.get_folder_by_server_relative_url(destination)
folder

In [ ]:
files = folder.files.execute_query().get()
files

In [ ]:
def sharepoint_entity_links(svg: str, spod: dict) -> str:
    result = svg
    for key, entity in spod['entities'].items():
        sp_id = entity['sharepoint']
        turl = f"https://fyayc.sharepoint.com/sites/CoPInformation-Data-Governance/Lists/Sandbox_Entities/DispForm.aspx?ID={sp_id}"
        result = result.replace(f'href="#{key}"', f'href="{turl}" target="_self" rel="noopener"')
    return result

In [ ]:
sharepoint_entity_links('<a href="#ENTI7022">bla ENTI7022<a/>', spod)

In [ ]:
def load_svg_content(key: str) -> str:
    folder = Path(configuration['content'], 'svg-diagrams-en')
    assert folder.is_dir()
    hits = list(folder.glob("*" + key + ".svg"))
    assert len(hits) > 0, f"No asset found in '{folder}' matching *-{key}.svg"
    svg_src_file = hits[0]
    assert svg_src_file.is_file()

    with open(svg_src_file, 'r') as src:
        svg_content = src.read()

    processed = sharepoint_entity_links(svg_content, spod)
    print(f"Rewrote size {len(svg_content)} -> {len(processed)}")
    outfile = Path(svg_src_file.parent, key + '.svg')
    with open(outfile, 'w') as out:
        out.write(processed)
    print(f"Wrote svg to {outfile}")
    
    return processed

In [ ]:
processed = load_svg_content('DIAG701')

In [ ]:
for key, diag in spod['diagrams'].items():
    processed = load_svg_content(key)
    files.upload(key + ".svg", processed.encode('utf-8'))
    ctx.execute_query()

In [ ]:
aspx_content = """<html><body>""" + processed + """</body></html>"""

In [ ]:
files.upload("DIAG700.aspx", aspx_content.encode('utf-8'))
ctx.execute_query()

In [ ]:
with open(Path(svg_src_file.parent, 'DIAG700.html'), 'w') as out:
    out.write(processed)

In [ ]:
spconf['content'] = "/Users/bue/dev/fyyccim-tools/content"

In [ ]:
folder = ctx.web.get_file_by_server_relative_path('Pages')

In [ ]:
folder.execute_query().get()
folder.properties

## Debugging

In [ ]:
entities = ctx.lists.get_by_title("Sandbox_Entities")

In [ ]:
entities.get().execute_query()
entities.item_count

In [ ]:
items = entities.items
items.get().execute_query()

In [ ]:
example_item = None
for item in items:
    if int(item.properties.get('Id')) == 76:
        print(f"item {item.properties.get('Id')} has value {item.properties}")
        print(f"{item.get_property('Diagrams')}")
        example_item = item

In [ ]:
example_item.set_property('Diagrams', 
                   f"""<div><a href="https://bossard.ch">Bossard</a></div>""")

In [ ]:
example_item.update().execute_query()